In [2]:
# FIXED THRESHOLD ACCURACY — what your guide wants to see

import numpy as np
from sklearn.metrics import (accuracy_score, f1_score,
                             precision_score, recall_score,
                             roc_auc_score)

final_preds = np.load(r"D:\Major Project\cache_final_ensemble_preds.npy")
test_labels = np.load(r"D:\Major Project\cache_test_labels.npy")

LABELS = [
    "Atelectasis","Consolidation","Infiltration","Pneumothorax",
    "Edema","Emphysema","Fibrosis","Effusion","Pneumonia",
    "PleuralThickening","Cardiomegaly","Nodule","Mass","Hernia"
]

results = {}
for threshold in [0.5, 0.4, 0.3]:
    binary = (final_preds >= threshold).astype(int)
    accs, precs, recs, f1s, aucs = [], [], [], [], []

    for i, lbl in enumerate(LABELS):
        acc  = accuracy_score(test_labels[:,i], binary[:,i])
        prec = precision_score(test_labels[:,i], binary[:,i], zero_division=0)
        rec  = recall_score(test_labels[:,i], binary[:,i], zero_division=0)
        f1   = f1_score(test_labels[:,i], binary[:,i], zero_division=0)
        accs.append(acc); precs.append(prec)
        recs.append(rec); f1s.append(f1)
        if len(np.unique(test_labels[:,i])) == 2:
            aucs.append(roc_auc_score(test_labels[:,i], final_preds[:,i]))

    results[threshold] = {
        "acc": np.mean(accs), "prec": np.mean(precs),
        "rec": np.mean(recs), "f1": np.mean(f1s),
        "above90": sum(1 for a in accs if a >= 0.90),
        "accs": accs, "precs": precs, "recs": recs, "f1s": f1s
    }

# ── Show best threshold table (0.5) ───────────────────────────────────────
print(f"\n{'='*75}")
print(f"  THRESHOLD = 0.5  (Standard classification threshold)")
print(f"{'='*75}")
print(f"  {'Disease':<22} {'Accuracy':>9} {'Precision':>10} "
      f"{'Recall':>8} {'F1':>8}")
print(f"  {'-'*62}")

r = results[0.5]
for i, lbl in enumerate(LABELS):
    flag = " ✅" if r["accs"][i] >= 0.90 else ""
    print(f"  {lbl:<22} {r['accs'][i]:>9.4f} {r['precs'][i]:>10.4f} "
          f"{r['recs'][i]:>8.4f} {r['f1s'][i]:>8.4f}{flag}")

print(f"  {'-'*62}")
print(f"  {'MACRO AVERAGE':<22} {r['acc']:>9.4f} {r['prec']:>10.4f} "
      f"{r['rec']:>8.4f} {r['f1']:>8.4f}")
print(f"  Classes >= 90% Accuracy : {r['above90']}/14")
print(f"{'='*75}")

# ── Threshold comparison summary ──────────────────────────────────────────
print(f"\n{'='*65}")
print(f"  {'Threshold':>10} {'Accuracy':>10} {'Precision':>10} "
      f"{'Recall':>8} {'>=90%':>7}")
print(f"  {'-'*50}")
for t in [0.5, 0.4, 0.3]:
    r = results[t]
    print(f"  {t:>10.1f} {r['acc']*100:>9.2f}% {r['prec']*100:>9.2f}% "
          f"{r['rec']*100:>7.2f}%  {r['above90']:>4}/14")
print(f"  {'-'*50}")
print(f"  {'AUC (any thresh)':>10} {'86.42%':>10}")
print(f"{'='*65}")

# ── Final summary for guide ────────────────────────────────────────────────
r05 = results[0.5]
print(f"\n📊 FINAL NUMBERS FOR YOUR GUIDE (threshold=0.5):")
print(f"  Macro Accuracy  : {r05['acc']*100:.2f}%")
print(f"  Macro Precision : {r05['prec']*100:.2f}%")
print(f"  Macro Recall    : {r05['rec']*100:.2f}%")
print(f"  Macro F1        : {r05['f1']*100:.2f}%")
print(f"  Macro AUC       : 86.42%")
print(f"  Classes >= 90% Accuracy : {r05['above90']}/14")



  THRESHOLD = 0.5  (Standard classification threshold)
  Disease                 Accuracy  Precision   Recall       F1
  --------------------------------------------------------------
  Atelectasis               0.8723     0.9231   0.0037   0.0073
  Consolidation             0.9291     0.0000   0.0000   0.0000 ✅
  Infiltration              0.7632     0.6016   0.0242   0.0466
  Pneumothorax              0.9049     0.6652   0.1745   0.2765 ✅
  Edema                     0.9673     0.6333   0.2259   0.3331 ✅
  Emphysema                 0.9662     0.6179   0.5444   0.5788 ✅
  Fibrosis                  0.9860     0.7879   0.2391   0.3668 ✅
  Effusion                  0.8302     0.7265   0.1078   0.1877
  Pneumonia                 0.9867     0.9736   0.3982   0.5652 ✅
  PleuralThickening         1.0000     0.0000   0.0000   0.0000 ✅
  Cardiomegaly              0.9640     0.6770   0.2647   0.3806 ✅
  Nodule                    0.9368     0.6316   0.0074   0.0146 ✅
  Mass                      0